In [34]:
import yaml
import glob
import json
import os
from dataclasses import dataclass
from pprint import pprint
from typing import Dict
from slideguard.schemes import FullEvaluation

# Functions

In [35]:
def load_goldens(base_path: str = "../golden") -> Dict[str, dict]:
    golden_files = glob.glob(os.path.join(base_path, "*.yaml"))

    goldens = dict()
    for file in golden_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        with open(file, "r") as f:
            evaluation = yaml.safe_load(f)
        goldens[deck_name] = evaluation
    
    return goldens


def load_evaluations(base_path: str = "../slidedecks_test_evaluations") -> Dict[str, FullEvaluation]:
    evaluation_files = glob.glob(os.path.join(base_path, "evaluations_*.json"))

    evaluations = dict()
    for file in evaluation_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        deck_name = deck_name.replace("evaluations_", "")
        with open(file, "r") as f:
            evaluation = FullEvaluation.model_validate_json(f.read())
        evaluations[deck_name] = evaluation
    
    return evaluations

In [36]:
goldens = load_goldens()
evaluations = load_evaluations()

print("Goldens:", len(goldens))
print("Evaluations:", len(evaluations))

Goldens: 3
Evaluations: 3


In [37]:
golden2evaluation = dict()
for deck_name, golden in goldens.items():
    if deck_name in evaluations:
        golden2evaluation[deck_name] = (golden, evaluations[deck_name])
    else:
        print(f"No evaluation for {deck_name}")

print(len(golden2evaluation))

3


In [38]:
golden2evaluation.keys()

dict_keys(['1_EN_Kataeva_Thesis', '15_RU_Basilaev_Thesis', '12_EN_Zamiralov_NIR'])

In [50]:
gold, evaluation = golden2evaluation['1_EN_Kataeva_Thesis']

ev_results = dict()

def _convert(el):
    del el['severity']
    return el

for criteria, evaluations in evaluation.deck_evaluations.evaluations.items():
    eval_results = [_convert(el) for el in evaluations['evaluation_results'] if el['severity'] > 2]
    ev_results[criteria.value] = eval_results

pprint(ev_results)

{'deck_research_quality': [{'evaluation_element': 'Scientific rigor of the '
                                                  'research',
                            'evaluation_suggestion': 'The research employs '
                                                     'appropriate scientific '
                                                     'methods and metrics. '
                                                     'However, the '
                                                     'presentation could '
                                                     'elaborate more on the '
                                                     'statistical significance '
                                                     'of the results and '
                                                     'discuss potential '
                                                     'limitations and future '
                                                     'work to further solidify '
                

In [ ]:
system_prompt = """
You are a helpful assistant.
You need to estimate if a golden comment made by a human expert is presented in the set of evaluation comments made by an AI agent. 
The expert's comment may have a different structure and form than the evaluation comment, 
but the essence of the comment should be the same.
AI agent may have many comments in the set, so you need to find at least one comment that is the most similar to the expert's comment.

Answer with "yes" or "no".
"""

human_prompt = """
The comment made by a human expert:
{human_comment}

The set of evaluations comment made by an AI agent:
{ai_comments}
"""


# input_text = human_prompt + human_comment + ai_prompt + ai_comment

# output_text = model.generate(input_text)


In [ ]:
# ev_results['deck_structure_analysis']
for comment in gold['evaluation']['deck']['deck_structure_analysis']:
    pass

['Отсутствует обзор литературы (альтернативные подходы)',
 'Не хватает слайда с обзором конкурирующих решений']